In [1]:
import numpy as np   
import pandas as pd      
import uproot as ur      
import time

#file = ur.open(r"data/last_Hijing_4M.root") #, note events in mult range 60-120 is about 640000 
#file = ur.open(r"data/merged_100_200.root")
file = ur.open(r"data/2merged_100_200.root")  
#file = ur.open(r"data/pp_isomerge_etapPb.root")    # v2=.05
#file = ur.open(r"data/pp_isomerge_v21_all.root")  #v2 = .1
  

# List all keys (e.g., trees, histograms)     
print(file.keys())    
   
# Access a TTree       
startt = 0 
endd = 1000000
tree = file["tree;1"]   #I have no clue why its jet_tree;1;1
phi  = tree['phi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
pt = tree['pt'].array(entry_start = startt, entry_stop = endd)
eta = tree['eta'].array(entry_start = startt, entry_stop = endd)
weights = tree['weight'].array(entry_start = startt, entry_stop = endd) 

# #phi = [dist.rvs(value=.05, size=len(x)) for x in phi]
phi = [np.array(x) for x in phi]
eta = [np.array(x) for x in eta]
pt = [np.array(x) for x in pt]
weights = weights/np.sum(weights)
testing12 = [[x, z, w, y] for x, z, w, y in zip(phi, weights, pt, eta)]  
# del pt
# # del file
# del eta
# del phi
# del file 

['tree;1']


In [2]:
#6 subevents implementation

import itertools
import numpy as np
from statsmodels.stats.weightstats import DescrStatsW as DL


# ================================================================
# Q-vector
# ================================================================

def Qmoment(a, n):
    return np.sum(np.exp(1j * n * a)).item()


# ================================================================
# Six-subevent differential correlators
#
# subevents:
#
# a = [-2.4, -1.6)
# b = [-1.6, -0.8)
# c = [-0.8,  0.0)
# d = [ 0.0,  0.8)
# e = [ 0.8,  1.6)
# f = [ 1.6,  2.4]
#
# For each possible POI subevent, this returns:
#
#   1 six-particle correlator
#   9 four-particle correlators
#   9 two-particle correlators
#
# Each entry is:
#
#   (correlator, weight)
#
# ================================================================

def sub6_diff6(phi, weight, pt, rapity, n,
               POI_start=1, POI_end=2):

    edges = [-2.4, -1.6, -0.8, 0.0, 0.8, 1.6, 2.4]

    labels = ['a', 'b', 'c', 'd', 'e', 'f']

    masks = {
        'a': (rapity >= edges[0]) & (rapity <  edges[1]),
        'b': (rapity >= edges[1]) & (rapity <  edges[2]),
        'c': (rapity >= edges[2]) & (rapity <  edges[3]),
        'd': (rapity >= edges[3]) & (rapity <  edges[4]),
        'e': (rapity >= edges[4]) & (rapity <  edges[5]),
        'f': (rapity >= edges[5]) & (rapity <= edges[6]),
    }

    M = {}
    m = {}
    Q = {}
    p = {}

    for lab in labels:

        phi_l = phi[masks[lab]]
        pt_l = pt[masks[lab]]

        # POI
        POI_l = phi_l[
            (pt_l >= POI_start) &
            (pt_l <= POI_end)
        ]

        # Reference particles
        phi_l = phi_l[pt_l < 3]

        M[lab] = len(phi_l)
        m[lab] = len(POI_l)

        Q[lab] = Qmoment(phi_l, n)
        p[lab] = Qmoment(POI_l, n)

    LEFT = ['a', 'b', 'c']
    RIGHT = ['d', 'e', 'f']

    def build(poi_label):

        # Require a POI in the POI subevent and at least
        # one reference particle in every other subevent.
        empty = (
            (m[poi_label] == 0) or
            any(
                M[lab] == 0
                for lab in labels
                if lab != poi_label
            )
        )

        if empty:

            keys = (
                ['Q6_abc_def']
                +
                [
                    f"Q4_{''.join(lp)}_{''.join(rp)}"
                    for lp in itertools.combinations(LEFT, 2)
                    for rp in itertools.combinations(RIGHT, 2)
                ]
                +
                [
                    f"Q2_{l}_{r}"
                    for l in LEFT
                    for r in RIGHT
                ]
            )

            return {
                k: (-1234.0, -1234.0)
                for k in keys
            }

        # POI subevent uses p,m.
        # All other subevents use Q,M.
        def val_and_mult(lab):

            if lab == poi_label:
                return p[lab], m[lab]

            return Q[lab], M[lab]

        out = {}

        # --------------------------------------------------------
        # Six-particle correlator
        #
        # <<6>> = < abc | def >
        # --------------------------------------------------------

        num = 1.0
        den = 1.0

        for lab in LEFT:

            q, mm = val_and_mult(lab)

            num *= q
            den *= mm

        for lab in RIGHT:

            q, mm = val_and_mult(lab)

            num *= np.conjugate(q)
            den *= mm

        out['Q6_abc_def'] = (
            np.real(num / den),
            den
        )

        # --------------------------------------------------------
        # Nine four-particle correlators
        #
        # <<4>> =
        #
        # ab|de
        # ab|df
        # ab|ef
        # ac|de
        # ac|df
        # ac|ef
        # bc|de
        # bc|df
        # bc|ef
        #
        # --------------------------------------------------------

        for l_pair in itertools.combinations(LEFT, 2):

            for r_pair in itertools.combinations(RIGHT, 2):

                num = 1.0
                den = 1.0

                for lab in l_pair:

                    q, mm = val_and_mult(lab)

                    num *= q
                    den *= mm

                for lab in r_pair:

                    q, mm = val_and_mult(lab)

                    num *= np.conjugate(q)
                    den *= mm

                key = f"Q4_{''.join(l_pair)}_{''.join(r_pair)}"

                out[key] = (
                    np.real(num / den),
                    den
                )

        # --------------------------------------------------------
        # Nine two-particle correlators
        #
        # <<2>> =
        #
        # a|d
        # a|e
        # a|f
        # b|d
        # b|e
        # b|f
        # c|d
        # c|e
        # c|f
        #
        # --------------------------------------------------------

        for l in LEFT:

            for r in RIGHT:

                ql, ml = val_and_mult(l)
                qr, mr = val_and_mult(r)

                out[f"Q2_{l}_{r}"] = (
                    np.real(
                        ql * np.conjugate(qr)
                        / (ml * mr)
                    ),
                    ml * mr
                )

        return out

    return (
        build('a'),
        build('b'),
        build('c'),
        build('d'),
        build('e'),
        build('f')
    )


# ================================================================
# Weighted average that safely removes -1234 entries
# ================================================================

def weighted_average(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (values != -1234) &
        (weights > 0)
    )

    if not np.any(mask):
        return np.nan, 0.0

    values = values[mask]
    weights = weights[mask]

    return (
        np.average(values, weights=weights),
        np.sum(weights)
    )


# ================================================================
# Differential d_n{6}
#
# d_n{6} =
#
# <<6'>>
# - 4 <<4'>> <<2>>
# - 2 <<4>> <<2'>>
# + 12 <<2'>> <<2>>^2
#
# For each possible POI subevent:
#
#   <<6'>>  = average of the 6-particle correlator
#
#   <<4'>>  = average of the 6 four-particle correlators
#              containing the POI subevent
#
#   <<4>>   = average of the 3 four-particle correlators
#              not containing the POI subevent
#
#   <<2'>>  = average of the 3 two-particle correlators
#              containing the POI subevent
#
#   <<2>>   = average of the 6 two-particle correlators
#              not containing the POI subevent
#
# ================================================================

def single_differential_cumulant6(dcor):

    LEFT = ['a', 'b', 'c']
    RIGHT = ['d', 'e', 'f']

    labels = ['a', 'b', 'c', 'd', 'e', 'f']

    # ------------------------------------------------------------
    # Determine which subevent contains the POI.
    #
    # We identify this from which Q4/Q2 terms contain a
    # differential weight corresponding to the POI.
    #
    # Here dcor is already constructed for one specific POI
    # subevent, so this function is called with poi_label.
    # ------------------------------------------------------------


def differential_cumulant6_for_poi(dcor, poi_label):

    LEFT = ['a', 'b', 'c']
    RIGHT = ['d', 'e', 'f']

    # ------------------------------------------------------------
    # Six-particle term
    # ------------------------------------------------------------

    d6, w6 = weighted_average(
        [dcor['Q6_abc_def'][0]],
        [dcor['Q6_abc_def'][1]]
    )

    # ------------------------------------------------------------
    # Four-particle terms
    #
    # 6 terms contain the POI subevent -> <<4'>>
    # 3 terms do not contain it      -> <<4>>
    # ------------------------------------------------------------

    d4_values = []
    d4_weights = []

    c4_values = []
    c4_weights = []

    for l_pair in itertools.combinations(LEFT, 2):

        for r_pair in itertools.combinations(RIGHT, 2):

            key = f"Q4_{''.join(l_pair)}_{''.join(r_pair)}"

            value, weight = dcor[key]

            if poi_label in l_pair or poi_label in r_pair:

                d4_values.append(value)
                d4_weights.append(weight)

            else:

                c4_values.append(value)
                c4_weights.append(weight)

    d4, wd4 = weighted_average(
        d4_values,
        d4_weights
    )

    c4, wc4 = weighted_average(
        c4_values,
        c4_weights
    )

    # ------------------------------------------------------------
    # Two-particle terms
    #
    # 3 terms contain the POI subevent -> <<2'>>
    # 6 terms do not contain it      -> <<2>>
    # ------------------------------------------------------------

    d2_values = []
    d2_weights = []

    c2_values = []
    c2_weights = []

    for l in LEFT:

        for r in RIGHT:

            key = f"Q2_{l}_{r}"

            value, weight = dcor[key]

            if poi_label == l or poi_label == r:

                d2_values.append(value)
                d2_weights.append(weight)

            else:

                c2_values.append(value)
                c2_weights.append(weight)

    d2, wd2 = weighted_average(
        d2_values,
        d2_weights
    )

    c2, wc2 = weighted_average(
        c2_values,
        c2_weights
    )

    # ------------------------------------------------------------
    # Check that everything exists
    # ------------------------------------------------------------

    if not (
        np.isfinite(d6) and 
        np.isfinite(d4) and
        np.isfinite(c4) and
        np.isfinite(d2) and
        np.isfinite(c2)
    ):

        return np.nan, 0.0

    # ------------------------------------------------------------
    # Differential sixth-order cumulant
    #
    # d_n{6} =
    #
    # <<6'>>
    # - 4 <<4'>> <<2>>
    # - 2 <<4>> <<2'>>
    # + 12 <<2'>> <<2>>^2
    #
    # ------------------------------------------------------------

    # dn6 = (
    #     d6
    #     - 4.0 * d4 * c2
    #     - 2.0 * c4 * d2
    #     + 12.0 * d2 * c2**2
    # )
    dn6 = ( #############################################eq 12 of https://journals.aps.org/prc/pdf/10.1103/PhysRevC.98.044902
        d6
        - 6.0 * d4 * c2
        - 3.0 * c4 * d2
        + 12.0 * d2 * c2**2
    )

    # The six-particle combination weight is used to combine
    # the six possible POI-subevent estimates.
    return dn6, w6

# ================================================================
# pt_binning
#
# FINAL OUTPUT ONLY:
#
#     dn6
#     dn6_standard_error
#
# ================================================================

def pt_binning(arr_list,
               mult_range,
               n,
               POI_start=1,
               POI_end=2,
               momentum_cut=0,
               nudge=False):

    arrs = np.array(arr_list, dtype=object)

    # ------------------------------------------------------------
    # Multiplicity selection
    #
    # Same convention as the existing pt_binning:
    # count particles with pt > 0.4
    # ------------------------------------------------------------

    lengths = np.array([
        np.sum(a[2] > 0.4)
        for a in arr_list
    ])

    mask = (
        (lengths >= mult_range[0]) &
        (lengths < mult_range[1])
    )

    arrs = arrs[mask]

    # ------------------------------------------------------------
    # Storage
    #
    # Six possible POI subevents.
    #
    # Each contains 19 correlator arrays:
    #
    #   1 six-particle
    #   9 four-particle
    #   9 two-particle
    # ------------------------------------------------------------

    labels = ['a', 'b', 'c', 'd', 'e', 'f']

    all_dcor = {
        lab: {
            'Q6_abc_def': [],
            **{
                f"Q4_{''.join(lp)}_{''.join(rp)}": []
                for lp in itertools.combinations(['a', 'b', 'c'], 2)
                for rp in itertools.combinations(['d', 'e', 'f'], 2)
            },
            **{
                f"Q2_{l}_{r}": []
                for l in ['a', 'b', 'c']
                for r in ['d', 'e', 'f']
            }
        }
        for lab in labels
    }

    all_weights = {
        lab: {
            key: []
            for key in all_dcor[lab]
        }
        for lab in labels
    }

    # ------------------------------------------------------------
    # Event loop
    # ------------------------------------------------------------

    for phi, weight, pt, eta in arrs:

        if nudge:
            # Keep this available for compatibility with the
            # existing framework. No nudge is applied here.
            pass

        dcor_a, dcor_b, dcor_c, dcor_d, dcor_e, dcor_f = sub6_diff6(
            phi,
            weight,
            pt,
            eta,
            n,
            POI_start=POI_start,
            POI_end=POI_end
        )

        dcor_list = [
            dcor_a,
            dcor_b,
            dcor_c,
            dcor_d,
            dcor_e,
            dcor_f
        ]

        for lab, dcor in zip(labels, dcor_list):

            for key in all_dcor[lab]:

                value, w = dcor[key]

                all_dcor[lab][key].append(
                    value
                )

                all_weights[lab][key].append(
                    w
                )

    del arrs

    # ------------------------------------------------------------
    # 20 round-robin blocks
    #
    # Same style as the existing pt_binning framework.
    # ------------------------------------------------------------

    n_splits = 20

    def round_robin_blocks(array, n_splits):

        array = np.asarray(array)

        return [
            array[k::n_splits]
            for k in range(n_splits)
        ]

    dcor_blocks = {
        lab: {
            key: round_robin_blocks(
                all_dcor[lab][key],
                n_splits
            )
            for key in all_dcor[lab]
        }
        for lab in labels
    }

    weight_blocks = {
        lab: {
            key: round_robin_blocks(
                all_weights[lab][key],
                n_splits
            )
            for key in all_weights[lab]
        }
        for lab in labels
    }

    # ------------------------------------------------------------
    # Calculate d_n{6} independently in every block
    # ------------------------------------------------------------

    dn6_blocks = []
    dn6_block_weights = []

    for k in range(n_splits):

        dn6_poi = []
        dn6_poi_weights = []

        # --------------------------------------------------------
        # Six possible POI subevents
        # --------------------------------------------------------

        for lab in labels:

            # Reconstruct the correlator dictionary for this block

            block_dcor = {}

            for key in dcor_blocks[lab]:

                block_dcor[key] = (
                    dcor_blocks[lab][key][k],
                    weight_blocks[lab][key][k]
                )

            dn6_poi_value, dn6_poi_weight = (
                differential_cumulant6_for_poi(
                    block_dcor,
                    lab
                )
            )

            if np.isfinite(dn6_poi_value) and dn6_poi_weight > 0:

                dn6_poi.append(
                    dn6_poi_value
                )

                dn6_poi_weights.append(
                    dn6_poi_weight
                )

        # --------------------------------------------------------
        # Weighted average over the six possible POI subevents
        # --------------------------------------------------------

        if len(dn6_poi) == 0:

            dn6_blocks.append(np.nan)
            dn6_block_weights.append(0.0)

        else:

            block_dn6, block_weight = weighted_average(
                dn6_poi,
                dn6_poi_weights
            )

            dn6_blocks.append(block_dn6)
            dn6_block_weights.append(block_weight)

    # ------------------------------------------------------------
    # Remove invalid blocks
    # ------------------------------------------------------------

    dn6_blocks = np.asarray(
        dn6_blocks,
        dtype=float
    )

    dn6_block_weights = np.asarray(
        dn6_block_weights,
        dtype=float
    )

    valid = (
        np.isfinite(dn6_blocks) &
        np.isfinite(dn6_block_weights) &
        (dn6_block_weights > 0)
    )

    dn6_blocks = dn6_blocks[valid]
    dn6_block_weights = dn6_block_weights[valid]

    # ------------------------------------------------------------
    # Final DL weighted average
    # ------------------------------------------------------------

    final_DL = DL(
        dn6_blocks,
        weights=dn6_block_weights
    )

    dn6 = final_DL.mean

    # Same standard-error convention used in your existing
    # pt_binning code:
    #
    # weighted std of the 20 block estimates / sqrt(20)
    #
    dn6_stderr = (
        final_DL.std /
        np.sqrt(len(dn6_blocks))
    )

    # ------------------------------------------------------------
    # ONLY OUTPUT
    # ------------------------------------------------------------

    return dn6, dn6_stderr

In [3]:
#no subevents implementation

def dcor_6(phi, pt, n, POI_start=1, POI_end=2):  # <6'> direct calculation
    # Differential 6-particle correlation from the Q-vector equation
    # Weights/normalization follow the direct-calculation formalism.

    phi = np.array(phi)
    pt = np.array(pt)

    # -------------------------
    # POI and reference particles
    # -------------------------
    mask = (pt >= POI_start) & (pt <= POI_end)
    POI = phi[mask]

    # Reference particles
    # Keep the ~mask if you want NO POI/reference overlap.
    # Remove ~mask if you want overlap allowed.
    mask_ref = (pt < reff_cut)
    Ref = phi[mask_ref]

    mp = len(POI)
    M = len(Ref)

    if (M <= 4) or (mp == 0):
        return -1234, -1234

    # -------------------------
    # Q-vectors for reference
    # -------------------------
    Qn = Qmoment(Ref, n)
    Q2n = Qmoment(Ref, 2*n)
    Q3n = Qmoment(Ref, 3*n)

    Qnc = np.conjugate(Qn)
    Q2nc = np.conjugate(Q2n)
    Q3nc = np.conjugate(Q3n)

    # -------------------------
    # p-vectors for POI
    # -------------------------
    pn = Qmoment(POI, n)

    # -------------------------
    # q-vectors = POI particles
    # that are also in the reference set
    # -------------------------
    mask_overlap = mask & mask_ref
    overlap = phi[mask_overlap]

    mq = len(overlap)

    qn = Qmoment(overlap, n)
    q2n = Qmoment(overlap, 2*n)
    q3n = Qmoment(overlap, 3*n)

    qnc = np.conjugate(qn)
    q2nc = np.conjugate(q2n)
    q3nc = np.conjugate(q3n)

    # ==========================================================
    # <6'>
    # ==========================================================

    term_p = (
        pn * Qn**2 * Qnc**3

        + pn * (
            -6*M * Qn * Qnc**2
            - Q2n * Qnc**3
            - 3 * Qn**2 * Qnc * Q2nc
            + 3 * Q2n * Qnc * Q2nc
            + 6*M**2 * Qnc
            + 18 * Qn * Qnc**2
            + 2 * Qn**2 * Q3nc
            - 2 * Q3nc * Q2n
            + 24 * Qnc
            - 30 * Qnc * M
            - 18 * Qn * Q2nc
            + 6*M * Qn * Q2nc
        )
    )

    term_qn = (
        qn * (
            12 * Qn * Qnc**2
            - 24 * M * Qnc
            - 12 * Qn * Q2nc
            + 96 * Qnc
        )
    )

    term_q2n = (
        q2n * (
            -2 * Qn * Qnc**3
            + 6*M * Qnc**2
            + 6 * Qn * Qnc * Q2nc
            - 6*M * Q2nc
            - 30 * Qnc**2
            - 4 * Q3nc * Qn
            + 30 * Q2nc
        )
    )

    term_qnc = (
        qnc * (
            6 * Qn**2 * Qnc
            - 12*M * Qn
            - 6 * Q2n * Qnc
            + 60 * Qn
        )
    )

    term_mq = (
        mq * (
            -3 * Qn**2 * Qnc**2
            + 12*M * Qn * Qnc
            + 3 * Q2n * Qnc**2
            + 3 * Qn**2 * Q2nc
            - 3 * Q2n * Q2nc
            - 6*M**2
            - 60 * Qn * Qnc
            + 54*M
            - 120
        )
    )

    term_q3n = (
        + 2 * q3n * Qnc**3
        - 6 * q3n * Q2nc * Qnc
        + 4 * q3n * Q3nc
        - 6 * q2nc * Qn**2
        + 6 * q2nc * Q2n
    )

    numerator = (
        term_p
        + term_qn
        + term_q2n
        + term_qnc
        + term_mq
        + term_q3n
    )
    denominator = (
        (mp*M-5*mq)
        * (M - 1)
        * (M - 2)
        * (M - 3)
        * (M - 4)
    )
    # corr6 = numerator / denominator

    return numerator / denominator, denominator
import numpy as np   
import pandas as pd     
import uproot as ur       
from statsmodels.stats.weightstats import DescrStatsW as DL 
reff_cut = 3

#def Qn
#paper 1 https://arxiv.org/pdf/1010.0233 
#paper 2 https://arxiv.org/pdf/1701.03830 
# Minee  
def Qmoment(a, n):    
    return np.sum(np.exp(1j*n*a)).item()  
    
# these functions now return the correlation and then the weight which is basically number of combinations
#coorelation 2, 4 with no subevents    
def corrilation_4(phi, n, pt):  #eq. 18, weights by eq. 10 
    #if (POI_true == True):
    #reff_cut = 3
    #mask = (pt>=POI_start) & (pt<= POI_end )
    mask_ref = (pt<reff_cut)# & (pt<=POI_start) | (pt>= POI_end )
    phi = phi[mask_ref] 
    M=len(phi) 
    if M<=3:
        #return np.nan 
        return -1234, -1234
    else: 
        Qn = Qmoment(phi, n) 
        Q2n = Qmoment(phi, 2*n)   
        demoninator = M*(M-1)*(M-2)*(M-3)
        first = np.abs(Qn)**4+np.abs(Q2n)**2-2*np.real(Q2n*np.conjugate(Qn)*np.conjugate(Qn))
        second = 2*(M-2)*np.abs(Qn)**2-M*(M-3)
        return (first-2*second)/demoninator, demoninator  
def corrilation_2(phi, n, pt ):   #eq. 16, weights by eq. 9
    #if (POI_true == True):
    #reff_cut = 3
    #mask = (pt>=POI_start) & (pt<= POI_end )
    mask_ref = (pt<reff_cut)
    phi = phi[mask_ref]
    M= len(phi)  
    if M<=1:
        #return np.nan
        return  -1234, -1234
    else:
        return (np.abs(Qmoment(phi, n))**2-M)/( (M-1)*M ), (M-1)*M
 
def Full_0sub(phi,pt,  n, POI_start=1 , POI_end = 2):
    #returns dcor4, dcor4w, dcor2, dcor2w
    #reff = 3
    
    #phi = phi[pt>momentum_cut]
    #def dcor_4(phi, pt, n, POI_cut): #equ 32, with, weights by eq. 25
    phi = np.array(phi); pt = np.array(pt)
    mask_POI = (pt>=POI_start) & (pt<= POI_end) 
    mask_Both = (pt>=POI_start) & (pt<= POI_end) & (pt<reff_cut) 
    POI = phi[mask_POI]
    Ref = phi[pt<reff_cut]
    Both = phi[mask_Both] 
    mp = len(POI); M=len(Ref); mq = len(Both)
    pn = Qmoment(POI, n)
    Qn = Qmoment(Ref, n); Qnc = np.conjugate(Qn)
    qn = Qmoment(Both,  n); q2n = Qmoment(Both, 2*n)
    if ( (mp==0 or M==0 )and mq==0):
        return -1234, -1234, -1234, -1234
    d2cor = (pn*Qnc-mq)/(mp*M-mq)
    d2corw = (mp*M-mq)
    if (M<3 or ((M==0 or mp==0) and mq==0)):
        return -1234, -1234, d2cor, d2corw
    d4cor =( (pn*Qn*Qnc*Qnc -q2n*Qnc*Qnc
             -pn*Qn*np.conjugate(Qmoment(Ref, 2*n)) -2*M*pn*Qnc
             -2*mq*np.abs(Qn)**2+7*qn*Qnc
             -Qn*np.conjugate(qn)+q2n*np.conjugate(Qmoment(Ref, 2*n))
             +2 *pn*Qnc+2*mq*M-6*mq
            )/( (mp*M-3*mq)*(M-1)*(M-2))       )
    d4corw = (mp*M-3*mq)*(M-1)*(M-2)
    return d4cor, d4corw, d2cor, d2corw       
    
def cors(phi, weight, pt,  n, POI_start=1 , POI_end = 2, momentum_cut = 0, nudge = False):
    # M = len(phi)
    if (nudge ==True):
        phi = nudge_v2(phi)
    # if (reff_cut<=POI_start):
    #     dcor2, dcor2w = dcor_2(phi, pt, n,  POI_start=POI_start,  POI_end = POI_end)
    #     dcor4, dcor4w = dcor_4(phi, pt, n,  POI_start=POI_start,  POI_end = POI_end)
    # else:
    dcor4, dcor4w, dcor2, dcor2w = Full_0sub(phi,pt,  n, POI_start=POI_start , POI_end = POI_end )
    d =  [dcor4, dcor4w, dcor2 , dcor2w ]
    cor2, cor2w = corrilation_2(phi, n, pt )
    cor4, cor4w = corrilation_4(phi, n, pt)
    c= [cor4, cor4w, cor2, cor2w]
    #phi = np.asarray(phi); pt = np.asarray(pt); mask = (pt>=POI_start) & (pt<= POI_end ); phii= phi[mask];fcor2, fcor2w = fake_corrilation_2(phii, n); fcor4,fcor4w = fake_corrilation_4(phii, n)
    return d, c #, [fcor4, fcor4w, fcor2, fcor2w]# dcor4, dcor2, cor4, cor2, fake shit


from collections import defaultdict    

#for older pt function with csv, see HIJing test
    
def pt_binning6(arr_list, mult_range, n, POI_start=1 , POI_end = 2, momentum_cut=0, nudge=False):
    arrs = np.array(arr_list, dtype=object)

    # Vectorized extraction of lengths
    #lengths = np.array([len(a[0]) for a in arr_list]) #lengths is mult
    lengths = np.array([np.sum(a[2]>.4) for a in arr_list]) #lengths is mult, only count particles when pt>.4 beacsue thats what cms does
    

    mask = (lengths >= mult_range[0]) & (lengths < mult_range[1]) 
    #print(mask)
    arrs = arrs[mask]
    print(len(arrs))
    

    # 10 lists for each subevent

    #these are components for differerential and standard cumulants for both 2, 4, 6 correlations, no subevents. only 6 differential
    dcore0 = [[] for _ in range(4)]
    core0 = [[] for _ in range(4)]
    d6core0 = [[] for _ in range(2)]


    
    #dist = MyPDF(a=-np.pi, b=np.pi) 
    #this for looops calculates all the needed correlators and weights. It then populates the storage functions
    for phi, weight, pt, eta in arrs:
        e, f = cors(phi, weight, pt, n,POI_start=POI_start, POI_end = POI_end) #returns  d =  [dcor4, dcor4w, dcor2 , dcor2w ], then d, c
        k,kk = dcor_6(phi, pt, n, POI_start=1, POI_end=2) #returns  d =  [dcor4, dcor4w, dcor2 , dcor2w ], then d, c
        for i in range(4):
            dcore0[i].append(np.real(e[i]))
            core0[i].append(np.real(f[i]))
        d6core0[0].append(np.real(k))
        d6core0[1].append(np.real(kk))
            # fcore0[i].append(np.real(r[i]))
    del arrs
    #following two functions calculate the dcummulants given the correlators. we then need to break out corelators blocks into 20 parts to feed to this
    def single_differential_cumulant4(dcore4, not_array = False): #calculates the differential four particle cummulant, with subevents, returns this and the weight
        #eq 7 in csm paper, equivilent for other POI subevents
        #remove -1234 before the average. 
        for i in range(0, 10, 2):
            if (not_array == True):
                dcore4[i] = np.array(dcore4[i])
                dcore4[i+1] = np.array(dcore4[i+1])
            mask = (dcore4[i]==-1234)
            dcore4[i] = dcore4[i][~mask]
            dcore4[i+1] = dcore4[i+1][~mask]    
        #mask = (dcor[0]==1234)
        # print(dcor4[1])
        
        A1 = np.average(dcore4[0], weights=dcore4[1])
        A2 = np.average(dcore4[2], weights=dcore4[3])
        A3 = np.average(dcore4[4], weights=dcore4[5])
        A4 = np.average(dcore4[6], weights=dcore4[7])
        A5 = np.average(dcore4[8], weights=dcore4[9])
        
        return A1 - A2*A3 - A4*A5, np.sum(dcore4[1])
    def final_differential_cumulant4(dcor4ae4, dcor4be4, dcor4ce4, dcor4de4,not_array = False):
        #eqivilent eq of eq 4.26 of wang paper for differential cumualnts and 4 instead of 3 subevents
        if (not_array==True):
            d_2a, wa = single_differential_cumulant4(dcor4ae4, not_array = True)
            d_2b, wb = single_differential_cumulant4(dcor4be4, not_array = True)
            d_2c, wc = single_differential_cumulant4(dcor4ce4, not_array = True)
            d_2d, wd = single_differential_cumulant4(dcor4de4, not_array = True )
        else:
            d_2a, wa = single_differential_cumulant4(dcor4ae4, not_array = False)
            d_2b, wb = single_differential_cumulant4(dcor4be4, not_array = False)
            d_2c, wc = single_differential_cumulant4(dcor4ce4, not_array = False)
            d_2d, wd = single_differential_cumulant4(dcor4de4, not_array = False )
        return (d_2a*wa+d_2b*wb+d_2c*wc+d_2d*wd)/(wa+wb+wc+wd) , wa+wb+wc+wd

        
    def single_differential_cumulant2(dcore2,not_array = False): #calculates the differential four particle cummulant, with 2 subevents, returns this and the weight
        #eq 4 in csm paper, equivilent for other POI subevents
        #remove -1234 before the average. 
        for i in range(0, 6, 2):
            if (not_array == True):
                dcore2[i] = np.array(dcore2[i])
                dcore2[i+1] = np.array(dcore2[i+1])
            mask = (dcore2[i]==-1234)
            dcore2[i] = dcore2[i][~mask]
            dcore2[i+1] = dcore2[i+1][~mask]    
        
        A1 = np.average(dcore2[0], weights=dcore2[1])
        A2 = np.average(dcore2[2], weights=dcore2[3])
        A3 = np.average(dcore2[4], weights=dcore2[5])
        return A1 - 2*A2*A3, np.sum(dcore2[1])

        
    def cumulants0(dcor, cor, not_array = False):  #calculates the cumulants for 0 subevents based on weights and corelators given. 
        #note dcor is in the shape [dcor4, dcor4w, dcor2 , dcor2w ], similar for cor
        # of sumulant formulas paper
        #dn{2} is given by eq 30
        #dn{4} by eq 34
        # cn{4} by eq 12, cn{2} by eq 11
        for i in range(0, 4, 2):
            if (not_array == True):
                dcor[i] = np.array(dcor[i])
                dcor[i+1] = np.array(dcor[i+1])
                cor[i] = np.array(cor[i])
                cor[i+1] = np.array(cor[i+1])
            mask = (dcor[i]==-1234)
            dcor[i] = dcor[i][~mask]
            dcor[i+1] = dcor[i+1][~mask] 
            mask = (cor[i]==-1234)
            cor[i] = cor[i][~mask]
            cor[i+1] = cor[i+1][~mask] 
        
        dn2 = np.average(dcor[2], weights=dcor[3])
        dn4 = np.average(dcor[0], weights=dcor[1])-2*np.average(dcor[2], weights=dcor[3])*np.average(cor[2], weights= cor[3])

        cn2 = np.average(cor[2], weights=cor[3])
        cn4 = np.average(cor[0], weights=cor[1])-2*np.average(cor[2], weights= cor[3])**2
        return dn4, dn2, cn4, cn2, np.sum(dcor[1]), np.sum(dcor[3]), np.sum(cor[1]), np.sum(cor[3])
        
    def diff6_cumulants(dcor, cor, d6cor):
        for i in range(0, 4, 2):
            mask = (dcor[i]==-1234)
            dcor[i] = dcor[i][~mask]
            dcor[i+1] = dcor[i+1][~mask] 
            mask = (cor[i]==-1234)
            cor[i] = cor[i][~mask]
            cor[i+1] = cor[i+1][~mask]
        mask = (d6cor[0]==-1234)
        d6cor[0] = d6cor[0][~mask]
        d6cor[1] = d6cor[1][~mask] 
        
        dn6 = (np.average(d6cor[0], weights=d6cor[1])-6*np.average(dcor[0], weights=dcor[1])*np.average(cor[2], weights=cor[3])-
               4*np.average(cor[0], weights=cor[1])*np.average(dcor[2], weights=dcor[3])+12*np.average(dcor[2], weights=dcor[3])*np.average(cor[2], weights=cor[3])**2)
        return dn6, np.sum(d6cor[1])
        
        

        

    #this breaks my corelator blocks into 20 separate parts, is hopefully faster than my slicing method
    #before dcor4ae4 was <4>_a', Weights for a, <2>_a',c, Weight, ...
    #now, dcor4ae4_blocks[i] is the ith split of this, so if dcor4ae4 has len 100. and dcor4ae4[0] = [2, 3, 4.5, ..., 4]
    # then dcor4ae4_blocks[0][0] = [2, 3, 4.5, ...] and dcor4ae4_blocks[19][0] = [..., 4]
    n_splits = 20

    def round_robin_blocks(arrays, n_splits):
        """
        Input:
            arrays = [array0, array1, ...]
        Output:
            blocks[k][i] = i-th array restricted to subsample k
        """
        arrays = [np.asarray(a) for a in arrays]
        return [
            [arr[k::n_splits] for arr in arrays]
            for k in range(n_splits)
        ]
    
    def leave_one_out_blocks(blocks, n_splits):
        """
        Input:
            blocks[k][i] = i-th array restricted to subsample k
        Output:
            loo[k][i] = i-th array built from ALL subsamples EXCEPT k
                        (i.e. the delete-one-block jackknife dataset)
        """
        n_arrays = len(blocks[0])
        loo = []
        for k in range(n_splits):
            combined = [
                np.concatenate([blocks[j][i] for j in range(n_splits) if j != k])
                for i in range(n_arrays)
            ]
            loo.append(combined)
        return loo
    
    def jackknife_std(theta_loo):
        """
        Standard delete-one-group jackknife std error.
        theta_loo[k] = estimate with block k left out.
        """
        theta_loo = np.asarray(theta_loo, dtype=float)
        K = len(theta_loo)
        theta_bar = np.mean(theta_loo)
        return np.sqrt((K - 1) / K * np.sum((theta_loo - theta_bar) ** 2))
    
    # full-sample point estimates (unchanged)
    
    # build the 20 round-robin blocks (same as before)
    dcore0_blocks   = round_robin_blocks(dcore0, n_splits);   del dcore0
    d6core0_blocks   = round_robin_blocks(d6core0, n_splits);   del d6core0
    core0_blocks    = round_robin_blocks(core0, n_splits);    del core0
    jack_d6core0 = []
    wjack_d6core0 = []

    for k in range(n_splits): 
        #dn4, dn2, cn4, cn2, wdn4, wdn2, wcn4, wcn2 = cumulants0(dcore0_blocks[k], core0_blocks[k])
        dn6, wdn6 = diff6_cumulants(dcore0_blocks[k], core0_blocks[k], d6core0_blocks[k])
        jack_d6core0.append(dn6)
        wjack_d6core0.append(wdn6)

        

    return [DL(jack_d6core0, weights = wjack_d6core0).mean, DL(jack_d6core0, weights = wjack_d6core0).std/np.sqrt(20)]#]#, np.mean(jack_dcor4e2), jackknife_std(jack_dcor4e2)]
 


In [31]:

pt_binning6(testing12, [50,200], 2, POI_start=1 , POI_end = 2, momentum_cut=0, nudge=False)

99977


[np.float64(5.964940174393845e-07), np.float64(5.906458597859183e-08)]

In [4]:
bin_size = 1 
n=2
# base_string="nudge_PPb"  
end_cols = 11 
mult_range = [100, 200 ]       
dn6e6 = np.zeros(100)  
dn6stde6 = np.zeros(100)
dn6e0 = np.zeros(100)  
dn6stde0 = np.zeros(100)



# dn4e2 = np.zeros(100)
# dn4stde2 = np.zeros(100)

pt_bins = np.zeros(100) 

#CMS_pt = CMS_data['pT_GeV_green'].to_numpy() 
CMS_pt = np.array([ 0.402,  0.713,  1.213,  1.717,  2.219,  2.718,  3.396,  4.414,
         5.426])#,  6.766,  8.82 , 10.86] )
#CMS_pt = np.array([  0.402,  0.713,  1.213,  1.717,  2.219,  2.718,  3.396,  4.414, 5.426,  6.766,])  #8.82 , 10.86, 12.5] )

POI_PT_EDGES = np.array([0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0]) 
# POI_PT_EDGES = np.array([0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10, 12, 14, 20]) 
# POI_PT_MEAN = np.array(
#     [0.402, 0.713, 1.213, 1.717, 2.219, 2.718, 3.396, 4.414,
#      5.426, 6.766, 8.82, 10.86, 12.881, 17.188]
# )
# POI_PT_EDGES = np.array([0.3, 0.5, 1.0])   
POI_PT_MEAN = np.array([0.402, 0.713, 1.213, 1.717, 2.219, 2.718, 3.396, 4.414,5.426, 6.766, 8.82, 10.86, 12.881, 17.188])
CMS_pt = POI_PT_EDGES   
for i in range(0, len(CMS_pt)-1): 
    POI_start = CMS_pt[i] 
    #?
    POI_end = CMS_pt[i+1]

    results = pt_binning(testing12, mult_range, n, POI_start=POI_start, POI_end = POI_end,  momentum_cut=0, nudge=False) 
    #print(results)
    #print(POI_start)
  
    dn6e6[i] = results[0]
    dn6stde6[i] = results[1]#/np.sqrt(20)  

    pt_bins[i] = POI_PT_MEAN[i]
    results = pt_binning6(testing12, mult_range, n, POI_start=POI_start, POI_end = POI_end,  momentum_cut=0, nudge=False) 
    #print(results)
    #print(POI_start)
  
    dn6e0[i] = results[0]
    dn6stde0[i] = results[1]#/np.sqrt(20)


dn6e6 = np.trim_zeros(dn6e6)
dn6stde6 = np.trim_zeros(dn6stde6)
dn6e0 = np.trim_zeros(dn6e0)
dn6stde0 = np.trim_zeros(dn6stde0)


pt_bins = np.trim_zeros(pt_bins)

arrays_dataframe = {
        "dn6e6": dn6e6,
        "dn6stde6": dn6stde6, 
        "dn6e0": dn6e0,
        "dn6stde0": dn6stde0, 
        "pt_bins": pt_bins 
    
    }
arrays_dataframe           


# Create a DataFrame by aligning arrays by index (shorter ones become NaN)
df = pd.DataFrame(dict((k, pd.Series(v)) for k, v in arrays_dataframe.items()))
df.to_csv("dn6_baseHijing.csv", index=False)    

999620
999620
999620
999620
999620
999620
999620
999620
999620
999620


In [8]:
#testing12, mult_range, n, POI_start=POI_start, POI_end = POI_end +

dn6, dn6_stderr = pt_binning(
    testing12,
    [100,200],
    2,
    POI_start=1,
    POI_end=2
)

In [19]:
#drafts of the differential 6 particle correlators
reff_cut = 3
def dcor_6(phi, pt, n, POI_start=1, POI_end=2):  # <6'> direct calculation
    # Differential 6-particle correlation from the Q-vector equation
    # Weights/normalization follow the direct-calculation formalism.

    phi = np.array(phi)
    pt = np.array(pt)

    # -------------------------
    # POI and reference particles
    # -------------------------
    mask = (pt >= POI_start) & (pt <= POI_end)
    POI = phi[mask]

    # Reference particles
    # Keep the ~mask if you want NO POI/reference overlap.
    # Remove ~mask if you want overlap allowed.
    mask_ref = (pt < reff_cut)
    Ref = phi[mask_ref]

    mp = len(POI)
    M = len(Ref)

    if (M <= 4) or (mp == 0):
        return -1234, -1234

    # -------------------------
    # Q-vectors for reference
    # -------------------------
    Qn = Qmoment(Ref, n)
    Q2n = Qmoment(Ref, 2*n)
    Q3n = Qmoment(Ref, 3*n)

    Qnc = np.conjugate(Qn)
    Q2nc = np.conjugate(Q2n)
    Q3nc = np.conjugate(Q3n)

    # -------------------------
    # p-vectors for POI
    # -------------------------
    pn = Qmoment(POI, n)

    # -------------------------
    # q-vectors = POI particles
    # that are also in the reference set
    # -------------------------
    mask_overlap = mask & mask_ref
    overlap = phi[mask_overlap]

    mq = len(overlap)

    qn = Qmoment(overlap, n)
    q2n = Qmoment(overlap, 2*n)
    q3n = Qmoment(overlap, 3*n)

    qnc = np.conjugate(qn)
    q2nc = np.conjugate(q2n)
    q3nc = np.conjugate(q3n)

    # ==========================================================
    # <6'>
    # ==========================================================

    term_p = (
        pn * Qn**2 * Qnc**3

        + pn * (
            -6*M * Qn * Qnc**2
            - Q2n * Qnc**3
            - 3 * Qn**2 * Qnc * Q2nc
            + 3 * Q2n * Qnc * Q2nc
            + 6*M**2 * Qnc
            + 18 * Qn * Qnc**2
            + 2 * Qn**2 * Q3nc
            - 2 * Q3nc * Q2n
            + 24 * Qnc
            - 30 * Qnc * M
            - 18 * Qn * Q2nc
            + 6*M * Qn * Q2nc
        )
    )

    term_qn = (
        qn * (
            12 * Qn * Qnc**2
            - 24 * M * Qnc
            - 12 * Qn * Q2nc
            + 96 * Qnc
        )
    )

    term_q2n = (
        q2n * (
            -2 * Qn * Qnc**3
            + 6*M * Qnc**2
            + 6 * Qn * Qnc * Q2nc
            - 6*M * Q2nc
            - 30 * Qnc**2
            - 4 * Q3nc * Qn
            + 30 * Q2nc
        )
    )

    term_qnc = (
        qnc * (
            6 * Qn**2 * Qnc
            - 12*M * Qn
            - 6 * Q2n * Qnc
            + 60 * Qn
        )
    )

    term_mq = (
        mq * (
            -3 * Qn**2 * Qnc**2
            + 12*M * Qn * Qnc
            + 3 * Q2n * Qnc**2
            + 3 * Qn**2 * Q2nc
            - 3 * Q2n * Q2nc
            - 6*M**2
            - 60 * Qn * Qnc
            + 54*M
            - 120
        )
    )

    term_q3n = (
        + 2 * q3n * Qnc**3
        - 6 * q3n * Q2nc * Qnc
        + 4 * q3n * Q3nc
        - 6 * q2nc * Qn**2
        + 6 * q2nc * Q2n
    )

    numerator = (
        term_p
        + term_qn
        + term_q2n
        + term_qnc
        + term_mq
        + term_q3n
    )
    denominator = (
        (mp*M-5*mq)
        * (M - 1)
        * (M - 2)
        * (M - 3)
        * (M - 4)
    )
    # corr6 = numerator / denominator

    return numerator / denominator, denominator

In [4]:
import numpy as np
import itertools

def dcor_6_long(phi, pt, n, POI_start=1, POI_end=2):
    phi = np.asarray(phi)
    pt = np.asarray(pt)

    POI = np.where((pt >= POI_start) & (pt <= POI_end))[0]
    Ref = np.where(pt < 3)[0]

    summ = 0.0
    count = 0

    for i in POI:
        # 2 "+" reference particles (j1,j2), 3 "-" reference particles (k1,k2,k3)
        # all distinct from each other AND from i
        for j1, j2, k1, k2, k3 in itertools.permutations(Ref, 5):
            if i in (j1, j2, k1, k2, k3):
                continue
            summ += np.cos(
                n * (
                    phi[i] + phi[j1] + phi[j2]
                    - phi[k1] - phi[k2] - phi[k3]
                )
            )
            count += 1

    if count == 0:
        return np.nan, 0
    return summ / count, count

In [21]:
reff_cut = 3
def Qmoment(a, n):    
    return np.sum(np.exp(1j*n*a)).item()  
print(dcor_6(phi[0][0:15], pt[0][0:15], 2, POI_start = .3, POI_end = 1))
print(dcor_6_long(phi[0][0:15], pt[0][0:15], 2, POI_start = .3, POI_end = 1))

(np.complex128(0.0025120796595232064+0.00015360990350665846j), 2642640)
(np.float32(0.0025119707), 2642640)


In [22]:
0.0025120796595232064-0.0025119707

1.0895952320651506e-07

In [ ]:
0.0025120797 
0.0025119707

In [12]:
#cloude code

"""
Q-vector formalism for differential multi-particle azimuthal correlators,
derived by "direct calculation" (Bilandzic-Snellings-Voloshin style) via
Moebius inversion on the set-partition lattice, and cross-checked against
a brute-force nested for-loop implementation.

This is a sympy-free rewrite of qvector_correlators.py. Symbolic expressions
are represented as plain dicts mapping a monomial (a sorted tuple of symbol
names, e.g. ('M','Qc2')) to an integer coefficient, instead of sympy
expressions. This is sufficient because every term produced by the partition
sum is already a single monomial (a product of one symbol per block) -- the
only "simplification" ever needed is summing coefficients of identical
monomials across different partitions, which a dict does natively.

Definitions
-----------
RP  : M reference particles, angles phi[0..M-1]
POI : m_p particles of interest in a kinematic bin, angles psi[0..m_p-1]
      (a particle can be BOTH a POI and an RP -- the "overlap" set,
      size m_q, tracked via overlap_map: poi_index -> rp_index)

Vectors (harmonic n, integer multiples k*n):
    Q_k  = sum_{RP}          exp(i k n phi)
    p    = sum_{POI}         exp(i n psi)         (POI always enters at +n)
    q_k  = sum_{overlap}     exp(i k n phi)

A differential 2k-particle correlator is specified by a list `exps` of
+-1's, one per particle slot: exps[0] is always the POI slot (+1 by
convention), exps[1:] are the RP slots. E.g.
    2-particle : [ 1,-1]
    4-particle : [ 1, 1,-1,-1]
    6-particle : [ 1, 1, 1,-1,-1,-1]

derive(exps, mode) returns a dict {monomial_tuple: coeff} representing:
    mode="full"  -> numerator = sum over all mutually-distinct-particle
                     tuples of the product of exp(i*exps[r]*n*angle_r)
    mode="count" -> denominator = number of such distinct tuples
so that   <correlator> = numerator / denominator.
"""

import itertools
from math import factorial
from collections import Counter
import numpy as np


# ---------------------------------------------------------------- #
# 1. set-partition enumeration + Moebius function of the partition
#    lattice (mu(0hat, pi) = prod_blocks (-1)^(b-1) (b-1)!)
# ---------------------------------------------------------------- #
def set_partitions(collection):
    collection = list(collection)
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i, subset in enumerate(smaller):
            yield smaller[:i] + [[first] + subset] + smaller[i + 1:]
        yield [[first]] + smaller


def mobius_0(block_sizes):
    c = 1
    for b in block_sizes:
        c *= (-1) ** (b - 1) * factorial(b - 1)
    return c


# ---------------------------------------------------------------- #
# 2. symbol name assigned to a block of slots forced onto one particle
#    (returns a plain string instead of a sympy.Symbol)
# ---------------------------------------------------------------- #
def block_symbol(block, exps, poi_idx=0, mode="full"):
    total = sum(exps[i] for i in block)
    contains_poi = poi_idx in block

    if mode == "count":
        if contains_poi and len(block) == 1:
            return 'mp'
        elif contains_poi:
            return 'mq'
        else:
            return 'M'

    # mode == "full"
    if contains_poi and len(block) == 1:
        return 'p'                       # p_n
    elif contains_poi:
        if total == 0:
            return 'mq'
        elif total > 0:
            return f'q{total}'            # q_{k n}
        else:
            return f'qc{-total}'          # conj(q_{k n})
    else:
        if total == 0:
            return 'M'
        elif total > 0:
            return f'Q{total}'            # Q_{k n}
        else:
            return f'Qc{-total}'          # conj(Q_{k n})


# ---------------------------------------------------------------- #
# 3. the derivation itself: Moebius inversion on the partition lattice
#    (this is the exact combinatorial identity behind moments<->cumulants,
#    applied here to "sum over distinct indices" <-> "products of power sums")
#
#    Expression representation: dict {monomial: coeff}
#    monomial = tuple of symbol-name strings, SORTED so that e.g. ('M','Q1')
#    and ('Q1','M') collapse to the same dict key (this is the sympy-expand
#    step -- combining identical monomials from different partitions).
# ---------------------------------------------------------------- #
def derive(exps, mode="full"):
    n = len(exps)
    terms = {}
    for part in set_partitions(range(n)):
        sizes = [len(b) for b in part]
        mu = mobius_0(sizes)
        monomial = tuple(sorted(block_symbol(b, exps, 0, mode) for b in part))
        terms[monomial] = terms.get(monomial, 0) + mu
    # drop any monomials that cancelled to zero
    return {mono: c for mono, c in terms.items() if c != 0}


def pretty(expr):
    """Human-readable string, e.g. '4*M - 2*Q1*Qc1 + ...' (for printing only)."""
    parts = []
    for mono, c in expr.items():
        if not mono:
            parts.append(f"{c}")
            continue
        counts = Counter(mono)
        factors = []
        for sym, e in counts.items():
            factors.append(sym if e == 1 else f"{sym}^{e}")
        parts.append(f"{c}*" + "*".join(factors))
    return " + ".join(parts) if parts else "0"


# ---------------------------------------------------------------- #
# 4. brute-force nested-loop reference implementation (ground truth)
#    (unchanged -- never depended on sympy)
# ---------------------------------------------------------------- #
def brute_force(exps, n_harm, phi, psi, overlap_map):
    """
    exps        : e.g. [1,1,1,-1,-1,-1] for the differential 6-particle case
    n_harm      : the harmonic n
    phi         : array of M RP angles
    psi         : array of m_p POI angles
    overlap_map : dict poi_index -> rp_index for particles that are both

    Returns (numerator, count) computed by literal nested loops over
    mutually distinct particles -- the direct definition of the correlator,
    no Q-vector shortcuts.
    """
    M = len(phi)
    mp = len(psi)
    n_rp_slots = len(exps) - 1

    total = 0j
    count = 0
    for i in range(mp):
        excl = overlap_map.get(i, -1)
        for combo in itertools.permutations(range(M), n_rp_slots):
            if excl in combo:
                continue
            ang = exps[0] * psi[i]
            for e, a in zip(exps[1:], combo):
                ang += e * phi[a]
            total += np.exp(1j * n_harm * ang)
            count += 1
    return total, count


# ---------------------------------------------------------------- #
# 5. helper: evaluate a derived expression (dict form) on a numeric event
# ---------------------------------------------------------------- #
def evaluate_formula(expr, n_harm, phi, psi, overlap_map, max_k=6):
    M = len(phi)
    mp = len(psi)
    mq = len(overlap_map)
    ov_idx = list(overlap_map.values())

    subs = {'M': M, 'mp': mp, 'mq': mq}
    for k in range(1, max_k + 1):
        Qk = np.sum(np.exp(1j * k * n_harm * phi))
        subs[f'Q{k}'] = Qk
        subs[f'Qc{k}'] = np.conj(Qk)
        if mq > 0:
            qk = np.sum(np.exp(1j * k * n_harm * phi[ov_idx]))
        else:
            qk = 0j
        subs[f'q{k}'] = qk
        subs[f'qc{k}'] = np.conj(qk)
    subs['p'] = np.sum(np.exp(1j * n_harm * psi))

    total = 0j
    for monomial, coeff in expr.items():
        term = complex(coeff)
        for sym in monomial:
            term *= subs[sym]
        total += term
    return total


# if __name__ == "__main__":
#     np.random.seed(7)
#     n_harm = 2
#     M = 8
#     phi = np.random.uniform(0, 2 * np.pi, M)
#     # partial overlap: POI 0,1 are also RP 0,1 ; POI 2,3 are independent
#     overlap_map = {0: 0, 1: 1}
#     psi = np.array([phi[0], phi[1],
#                      np.random.uniform(0, 2 * np.pi),
#                      np.random.uniform(0, 2 * np.pi)])

#     for label, exps in [("2-particle", [1, -1]),
#                          ("4-particle", [1, 1, -1, -1]),
#                          ("6-particle", [1, 1, 1, -1, -1, -1])]:
#         num_expr = derive(exps, "full")
#         den_expr = derive(exps, "count")

#         num_formula = evaluate_formula(num_expr, n_harm, phi, psi, overlap_map)
#         den_formula = evaluate_formula(den_expr, n_harm, phi, psi, overlap_map).real

#         num_brute, cnt_brute = brute_force(exps, n_harm, phi, psi, overlap_map)

#         print(f"--- {label} ---")
#         print("  formula  :", pretty(num_expr))
#         print("  numerator diff  :", abs(num_formula - num_brute))
#         print("  denominator     :", den_formula, "vs brute count", cnt_brute)
#         print()
POI_start, POI_end = .3, 1
ref_cut = 3.0

pt = pt[0][0:15]
phi = phi[0][0:15]
ref_idx = np.where(pt < ref_cut)[0]          # reference particle indices
poi_idx = np.where((pt >= POI_start) & (pt <= POI_end))[0]  # POI indices

phi_rp = phi[ref_idx]                         # RP angles for Q-vectors
psi_poi = phi[poi_idx]                        # POI angles for p-vector

# overlap: for each POI, find its position within ref_idx if it's also a RP
overlap_map = {}
ref_pos = {orig: pos for pos, orig in enumerate(ref_idx)}
for p, orig in enumerate(poi_idx):
    if orig in ref_pos:
        overlap_map[p] = ref_pos[orig]
        
num_expr = derive([1, 1, 1, -1, -1, -1], "full")
den_expr = derive([1, 1, 1, -1, -1, -1], "count")

num = evaluate_formula(num_expr, n_harm=2, phi=phi_rp, psi=psi_poi,
                        overlap_map=overlap_map, max_k=6)
den = evaluate_formula(den_expr, n_harm=2, phi=phi_rp, psi=psi_poi,
                        overlap_map=overlap_map, max_k=6).real

print("6-particle correlator:", (num/den).real, den)

6-particle correlator: 0.0025120797 2642640.0


In [18]:
def pretty(expr):
    """Human-readable string, e.g. '4*M - 2*Q1*Qc1 + ...' (for printing only)."""
    parts = []
    for mono, c in expr.items():
        if not mono:
            parts.append(f"{c}")
            continue
        counts = Counter(mono)
        factors = []
        for sym, e in counts.items():
            factors.append(sym if e == 1 else f"{sym}^{e}")
        parts.append(f"{c}*" + "*".join(factors))
    return " + ".join(parts) if parts else "0"
num_expr = derive([1, 1, 1, -1, -1, -1], "full")
print(pretty(num_expr))

-120*mq + 24*Qc1*p + 30*Qc2*q2 + 60*Q1*qc1 + -18*Q1*Qc2*p + 4*Qc3*q3 + 6*Q2*qc2 + -2*Q2*Qc3*p + -4*Q1*Qc3*q2 + -6*Q1^2*qc2 + 2*Q1^2*Qc3*p + 96*Qc1*q1 + 54*M*mq + -30*M*Qc1*p + -6*M*Qc2*q2 + -12*Q1*Qc2*q1 + -12*M*Q1*qc1 + 6*M*Q1*Qc2*p + -30*Qc1^2*q2 + -60*Q1*Qc1*mq + 18*Q1*Qc1^2*p + -6*Qc1*Qc2*q3 + -3*Q2*Qc2*mq + -6*Q2*Qc1*qc1 + 3*Q2*Qc1*Qc2*p + 6*Q1*Qc1*Qc2*q2 + 3*Q1^2*Qc2*mq + 6*Q1^2*Qc1*qc1 + -3*Q1^2*Qc1*Qc2*p + -24*M*Qc1*q1 + -6*M^2*mq + 6*M^2*Qc1*p + 6*M*Qc1^2*q2 + 12*Q1*Qc1^2*q1 + 12*M*Q1*Qc1*mq + -6*M*Q1*Qc1^2*p + 2*Qc1^3*q3 + 3*Q2*Qc1^2*mq + -1*Q2*Qc1^3*p + -2*Q1*Qc1^3*q2 + -3*Q1^2*Qc1^2*mq + 1*Q1^2*Qc1^3*p


In [11]:
(0.000309865700133846-0.00029902646)/0.00029902646

0.03624843143929815

In [6]:
pt[0][0:15]

array([1.0290438 , 0.77720594, 1.0471514 , 0.673342  , 1.3149937 ,
       0.36287928, 1.2666619 , 0.71452355, 0.39845502, 0.48478308,
       0.7257799 , 0.41476718, 0.43742996, 0.35864705, 0.63014203],
      dtype=float32)

In [2]:
len({1,2,3,3})

3

In [ ]:
basic of 4 particle differential cummulant with for loops, that deals with overlap

def dcor_4_long(phi, pt, n, POI_start=1, POI_end=2):
    phi = np.asarray(phi)
    pt = np.asarray(pt)

    # Particle indices
    POI = np.where(
        (pt >= POI_start) & (pt <= POI_end)
    )[0]

    # Reference particles -- DO NOT remove the POIs
    Ref = np.where(pt < 3)[0]

    summ = 0.0
    count = 0

    for i in POI:
        for j in Ref:
            for k in Ref:
                for m in Ref:

                    # All four particles must be different
                    if len({i, j, k, m}) < 4:
                        continue

                    summ += np.cos(
                        n * (
                            phi[i]
                            + phi[j]
                            - phi[k]
                            - phi[m]
                        )
                    )

                    count += 1

    if count == 0:
        return np.nan, 0

    return summ / count, count

In [9]:
dn6

np.float64(-3.0585771881547e-08)

In [ ]:
#base 

import numpy as np   
import pandas as pd     
import uproot as ur       
from statsmodels.stats.weightstats import DescrStatsW as DL 
reff_cut = 3

#def Qn
#paper 1 https://arxiv.org/pdf/1010.0233 
#paper 2 https://arxiv.org/pdf/1701.03830 
# Minee  
def Qmoment(a, n):    
    return np.sum(np.exp(1j*n*a)).item()  
    



import itertools
import numpy as np

# ----------------------------------------------------------------------
# c_n{6} for 6 subevents, split LEFT={a,b,c} | RIGHT={d,e,f}:
#
# c_n{6}_{abc|def} = <<6>>_{abc|def}
#   - [ <<4>>_{ab|de}<<2>>_{c|f} + <<4>>_{ab|df}<<2>>_{c|e} + <<4>>_{ac|de}<<2>>_{b|f}
#     + <<4>>_{ac|df}<<2>>_{b|e} + <<4>>_{bc|de}<<2>>_{a|f} + <<4>>_{bc|df}<<2>>_{a|e}
#     + <<4>>_{ab|ef}<<2>>_{c|d} + <<4>>_{ac|ef}<<2>>_{b|d} + <<4>>_{bc|ef}<<2>>_{a|d} ]
#   + 2[ <<2>>_{a|d}<<2>>_{b|e}<<2>>_{c|f} + <<2>>_{a|d}<<2>>_{b|f}<<2>>_{c|e}
#      + <<2>>_{a|e}<<2>>_{b|d}<<2>>_{c|f} + <<2>>_{a|e}<<2>>_{b|f}<<2>>_{c|d}
#      + <<2>>_{a|f}<<2>>_{b|d}<<2>>_{c|e} + <<2>>_{a|f}<<2>>_{b|e}<<2>>_{c|d} ]
#
# Building blocks needed (all with "before | = unconjugated, after | =
# conjugated", exactly like the ab|cd convention in sub4_diff_cors):
#   - 1  six-particle term:  abc|def
#   - 9  four-particle terms: {ab,ac,bc} x {de,df,ef}
#   - 9  two-particle terms:  {a,b,c} x {d,e,f}
# = 19 unique (value, weight) terms total.
#
# This function only builds those 19 raw, per-event terms (numerator and
# its multiplicity-product weight) for each of the 6 possible POI
# subevents -- exactly what sub4_diff_cors does for c_n{4}. It does NOT
# do the event-averaging or the final formula combination above; do that
# afterward: average (value*weight) and weight separately across events
# (weighted mean) to get each <<m>>, then plug into the formula.
# ----------------------------------------------------------------------
def sub6_diff6(phi, weight, pt, rapity, n, POI_start=1, POI_end=2):
    edges = [-2.4, -1.6, -0.8, 0.0, 0.8, 1.6, 2.4]
    labels = ['a', 'b', 'c', 'd', 'e', 'f']
    masks = {
        'a': (rapity >= edges[0]) & (rapity <  edges[1]),
        'b': (rapity >= edges[1]) & (rapity <  edges[2]),
        'c': (rapity >= edges[2]) & (rapity <  edges[3]),
        'd': (rapity >= edges[3]) & (rapity <  edges[4]),
        'e': (rapity >= edges[4]) & (rapity <  edges[5]),
        'f': (rapity >= edges[5]) & (rapity <= edges[6]),
    }

    M, m, Q, p = {}, {}, {}, {}
    for lab in labels:
        phi_l = phi[masks[lab]]
        pt_l = pt[masks[lab]]
        POI_l = phi_l[(pt_l >= POI_start) & (pt_l <= POI_end)]
        phi_l = phi_l[pt_l < 3]  # RFPs, same cut as sub4_diff_cors

        M[lab] = len(phi_l)
        m[lab] = len(POI_l)
        Q[lab] = Qmoment(phi_l, n)
        p[lab] = Qmoment(POI_l, n)

    LEFT, RIGHT = ['a', 'b', 'c'], ['d', 'e', 'f']

    def build(poi_label):
        # mirrors the -1234 sentinel in sub4_diff_cors: bail if the POI
        # subevent has no POI particles, or any OTHER subevent is empty
        empty = (m[poi_label] == 0) or any(M[lab] == 0 for lab in labels if lab != poi_label)
        if empty:
            keys = (['Q6_abc_def']
                     + [f"Q4_{''.join(lp)}_{''.join(rp)}"
                        for lp in itertools.combinations(LEFT, 2)
                        for rp in itertools.combinations(RIGHT, 2)]
                     + [f"Q2_{l}_{r}" for l in LEFT for r in RIGHT])
            return {k: (-1234, -1234) for k in keys}

        def val_and_mult(lab):
            # POI subevent uses (p, m); everyone else uses (Q, M)
            return (p[lab], m[lab]) if lab == poi_label else (Q[lab], M[lab])

        out = {}

        # ---- 6-particle term: abc|def ----
        num, den = 1.0, 1.0
        for lab in LEFT:
            q, mm = val_and_mult(lab); num *= q; den *= mm
        for lab in RIGHT:
            q, mm = val_and_mult(lab); num *= np.conjugate(q); den *= mm
        out['Q6_abc_def'] = (num / den, den)

        # ---- 9 four-particle terms: {ab,ac,bc} x {de,df,ef} ----
        for l_pair in itertools.combinations(LEFT, 2):
            for r_pair in itertools.combinations(RIGHT, 2):
                num, den = 1.0, 1.0
                for lab in l_pair:
                    q, mm = val_and_mult(lab); num *= q; den *= mm
                for lab in r_pair:
                    q, mm = val_and_mult(lab); num *= np.conjugate(q); den *= mm
                key = f"Q4_{''.join(l_pair)}_{''.join(r_pair)}"
                out[key] = (num / den, den)

        # ---- 9 two-particle terms: {a,b,c} x {d,e,f} ----
        for l in LEFT:
            for r in RIGHT:
                ql, ml_ = val_and_mult(l)
                qr, mr_ = val_and_mult(r)
                out[f"Q2_{l}_{r}"] = (ql * np.conjugate(qr) / (ml_ * mr_), ml_ * mr_)

        return out

    # one dict of 19 named (value, weight) terms per POI subevent
    dcor6a = build('a')
    dcor6b = build('b')
    dcor6c = build('c')
    dcor6d = build('d')
    dcor6e = build('e')
    dcor6f = build('f')

    return dcor6a, dcor6b, dcor6c, dcor6d, dcor6e, dcor6f 



from collections import defaultdict    

# 